![WEkEO by Copernicus and Partners](https://github.com/wekeo/frameworks/blob/main/img/WEkEO_banner_EUMETSAT.png?raw=true)

<div style="display: flex; justify-content: space-between;">
  <a href="./01_Introduction_to_EOCanvas.ipynb">&lt;&lt; Introduction to the EOCanvas</a>
  <a href="./03_EOCanvas_DataTailor.ipynb">Using the Data Tailor EOCanvas function &gt;&gt;</a>
</div>

Copyright (c) 2024 European Union 

License: MIT

# Introducing the EOCanvas API functions

**This notebook can be run on the WEkEO JupyterHub. You can also view it on GitHub.**

[![launch-WEKEO](https://img.shields.io/badge/launch-WEKEO-1a4696.svg)](https://jupyterhub.prod.wekeo2.eu/hub/user-redirect/lab/tree/public/wekeo4data/wekeo-eocanvas/02_Introduction_to_EOCanvas_API.ipynb)
[![github](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/wekeo/wekeo-eocanvas/blob/main/02_Introduction_to_EOCanvas_API.ipynb)

**PREREQUISITES**

This notebook has the following prerequisites:
- [A WEkEO account](https://my.wekeo.eu/user-registration)
- Execution of the [EOCanvas Introduction Notebook](https://github.com/wekeo/wekeo4data/blob/main/wekeo-eocanvas/01_Introduction_to_EOCanvas.ipynb)

**LEARNING OUTCOMES**

At the end of this notebook you will know:

- how to list available processes of the EOCanvas
- how to list the history of executed processes
- how to get job logs 
- how to manually download processing results

**CONTENTS**

1. [Setting up](#1-setting-up)  
2. [Explore the functions of the API](#2-explore-the-functions-of-the-api)  
3. [Take home messages](#3-take-home-messages)  
4. [Next steps](#4-next-steps)

<hr>

The The EOCanvas is a WEkEO service to process Coperncius data in the Cloud. This notebooks gives a deep dive into more useful functions of the EOCanvas API to help you work with the EOCanvas module.

## 1. Setting up

Install eocanvas to your environment.

In [ ]:
! pip install -q eocanvas

Loadning necessary libraries

In [1]:
from eocanvas import API, Credentials
from eocanvas.api import Input, Config, ConfigOption
from eocanvas.processes import SnapProcess
from eocanvas.snap.graph import Graph
import os
import getpass
import hda
from pathlib import Path

### Setup the WEkEO HDA and credentials

To use the WEkEO HDA adaptor, create a file called `.hdarc` in your home directory:

Add the following lines to the file:

`user:<your_user_name>`

`password:<your_password>`

For more information about the HDA and how to set up your credentials, check out [this notebook](https://github.com/wekeo/wekeo-hda/blob/master/wekeo_harmonised_data_access_api.ipynb).

The EOCanvas reads your WEkEO credentials from the same `.hdarc` file.

Lets create our credentials file, if it doesn't already exist.

In [ ]:
 # load credentials
wekeo_credentials_file = Path(Path.home() / '.hdarc' )

def get_credentials():
    user = input('Enter your WEkEO user name: ').strip()
    password = getpass.getpass('Enter your WEkEO password: ').strip()
    return user, password

def save_credentials(user, password):
    os.makedirs(os.path.dirname(wekeo_credentials_file), exist_ok=True)
    with open(wekeo_credentials_file, "w", encoding="utf-8") as f:
        f.write(f'user: {user}\npassword: {password}')

# Try loading saved credentials
if not wekeo_credentials_file.is_file():
    user, password = get_credentials()
    save_credentials(user, password)

# Try to get token, retry if credentials are invalid
while True:
    try:
        token = hda.Client().token
        print(f"Your WEkEO token: '{token}'")
        break  # Exit loop if successful
    except hda.api.ConfigurationError as e:
        print("Invalid username or password. Please try again.")
        print (e)
        user, password = get_credentials()
        save_credentials(user, password)

## 2. Explore the functions of the API

You can get more information about the processes of the serverless function using the following commands: 

In [2]:
# initialize the API

api = API()

### 2.1 Which processes are available in the serverless functions?

In [3]:
processes = api.get_processes()
processes

[Process(api=<eocanvas.api.API object at 0x000001EBAD06F8B0>, process_id='snap-function', version='10.0.0', title='snap-function', description='Function based on the ESA SNAP tool version 10.', inputs={'eo_config': {'title': 'Configuration for eo_input', 'description': 'Configuration for the eo_input in JSON format. They instruct the function on how to deal with the eo_input (e.g., uncompress).', 'minOccurs': 0, 'maxOccurs': 1, 'schema': {'$ref': 'https://string'}}, 'snap_graph': {'title': 'ESA SNAP GPT graph', 'description': 'A base64-encoded gpt graph. It should contain placeholders referring to the eo_input field.', 'minOccurs': 0, 'maxOccurs': 1, 'schema': {'$ref': 'https://string'}}, 'eo_input': {'title': 'eo_inputs Title', 'description': "Named inputs provided to the ESA SNAP tool in JSON encoding. They should match the placeholders in the SNAP's GPT graph.", 'minOccurs': 0, 'maxOccurs': 1, 'schema': {'$ref': 'http://named_inputs'}}}, output=None),
 Process(api=<eocanvas.api.API 

You can see that two processes are available as of now: 

1. the **dataTailor** function and 
2. the **snap** function

Along with the definition of the functions, you see their description and the inputs each function takes. 

### 2.2 Which processes did I submit to the EOCanvas?

Using `get_job()` the api will return all of the submitted jobs. 

In [4]:
jobs = api.get_jobs()
jobs

[Job(api=<eocanvas.api.API object at 0x000001EBAD06F8B0>, job_id='eb84e267-08c6-516a-974e-d90cb5b621ee', status='successful', started='2025-09-30 06:54:10', created='2025-09-30 06:54:10', updated='2025-09-30 06:56:15', finished='2025-09-30 06:56:15'),
 Job(api=<eocanvas.api.API object at 0x000001EBAD06F8B0>, job_id='f090c2e4-149b-5d8b-b109-b2fd3d30625e', status='successful', started='2025-08-05 13:31:20', created='2025-08-05 13:31:20', updated='2025-08-05 13:33:26', finished='2025-08-05 13:33:26'),
 Job(api=<eocanvas.api.API object at 0x000001EBAD06F8B0>, job_id='f53439e2-4e24-53f7-bb02-c15ad3687e89', status='successful', started='2025-07-16 14:46:27', created='2025-07-16 14:46:27', updated='2025-07-16 14:50:23', finished='2025-07-16 14:50:22'),
 Job(api=<eocanvas.api.API object at 0x000001EBAD06F8B0>, job_id='c636a2f1-0893-5a9c-9333-b58c83cf61d9', status='successful', started='2025-07-16 13:30:16', created='2025-07-16 13:30:16', updated='2025-07-16 13:34:38', finished='2025-07-16 13:3

### 2.3 How do I get the job logs of one specific job?

In [7]:
# select the job you want to get the logs for

job = jobs[0].job_id

In [8]:
api.get_job_logs(job=job)

[LogEntry(timestamp=datetime.datetime(2025, 9, 30, 6, 54, 29, 371997, tzinfo=datetime.timezone.utc), message='time="2025-09-30T06:54:29.371Z" level=info msg="Starting Workflow Executor" version=v3.6.10'),
 LogEntry(timestamp=datetime.datetime(2025, 9, 30, 6, 54, 29, 377253, tzinfo=datetime.timezone.utc), message='time="2025-09-30T06:54:29.377Z" level=info msg="Using executor retry strategy" Duration=1s Factor=1.6 Jitter=0.5 Steps=5'),
 LogEntry(timestamp=datetime.datetime(2025, 9, 30, 6, 54, 29, 377292, tzinfo=datetime.timezone.utc), message='time="2025-09-30T06:54:29.377Z" level=info msg="Executor initialized" deadline="0001-01-01 00:00:00 +0000 UTC" includeScriptOutput=false namespace=ws-serverless podName=workflow-5frd5-stage-in-3066650963 templateName=stage-in version="&Version{Version:v3.6.10,BuildDate:2025-06-04T12:47:02Z,GitCommit:459c19db6e9dd86dd757c21644404cb784863fae,GitTag:v3.6.10,GitTreeState:clean,GoVersion:go1.24.3,Compiler:gc,Platform:linux/amd64,}"'),
 LogEntry(timesta

You can now see the full logs of the process. 

### 2.4 How do I download the results from a job which has been launched in the past?

To download the results again, you can manually trigger the download using the job id.

In [9]:
#get and inspect the results
results = api.get_job_results(job=job)[0]
results

Result(api=<eocanvas.api.API object at 0x000001EBAD06F8B0>, href='/download/result/17962', title='wks3:///eb84e267-08c6-516a-974e-d90cb5b621ee/output/6c2a2db4-9dca-11f0-9901-0a5e9884a670.nc', rel='enclosure')

In [10]:
results.download(download_dir="result")

## 3. Take home messages

- The `API` object lists the available EOCanvas processes and the jobs you have submitted.
- Job logs help you to monitor and debug a specific job.
- The results of past jobs can be downloaded again at any time using the job ID.

## 4. Next steps

This notebook provided an overview of the basic functionality of the EOCanvas Serverless Functions API. For more applied examples of what you can do with the EOCanvas you can go to the next notebook.

<hr>

[View on GitHub](https://github.com/wekeo) | 
[Visit WEkEO](https://wekeo.copernicus.eu) | [Run Notebooks on the WEkEO Workspace](https://jupyterhub.prod.wekeo2.eu/) | 
[Contact helpdesk for support](mailto:support@wekeo.eu)